# QickSim RFDC Output Waveform Demo

This notebook demonstrates an `AveragerProgram`-style AWG tuning program and plots the RFDC-facing digital output streams discovered from the HWH topology, without PYNQ hardware, RFDC hardware, Vivado, or bitstream download.

The output samples shown here are the packed/lane samples at the RFDC AXIS DAC inputs. They are not analog RFDC output waveforms.

In [ ]:
from pathlib import Path
import sys

REPO = Path(r"C:\JeonghyunPark\Workspace\QSTL_QICK")
QICK_LIB = REPO / "qick" / "qick_lib"
if str(QICK_LIB) not in sys.path:
    sys.path.insert(0, str(QICK_LIB))

from qick import AveragerProgram
from qick.sim import QickSim

print("QICK lib:", QICK_LIB)

## Configuration

All clock values are in MHz. `dac_clk` is the RFDC DAC sample rate, `fabric_clk` is the RFDC/generator fabric clock, and `tproc_clk` is the tProcessor timing clock.

In [ ]:
BITFILE = REPO / "qick" / "firmware" / "projects" / "qstl_awg_tuning" / "bitstream.bit"
HWHFILE = None

DAC_CLK = 6144.0
ADC_CLK = 4096.0
FABRIC_CLK = 384.0
TPROC_CLK = 250.0
CYCLES = 256
PLOT_SAMPLES = 4096

if not BITFILE.exists():
    raise FileNotFoundError(f"bitfile does not exist: {BITFILE}")

BITFILE

In [ ]:
sim = QickSim(
    BITFILE,
    hwhfile=HWHFILE,
    strict=False,
    dac_clk=DAC_CLK,
    adc_clk=ADC_CLK,
    fabric_clk=FABRIC_CLK,
    tproc_clk=TPROC_CLK,
)

print(f"gens: {len(sim['gens'])}")
print(f"awg_tunings: {len(sim['awg_tunings'])}")
print("DAC 00 config:", sim["rf"]["dacs"].get("00"))
print("tProc config:", sim["tprocs"][0] if sim["tprocs"] else None)

## Discovered RFDC-facing generators

In [ ]:
for idx, gen in enumerate(sim["gens"]):
    print(
        f"gen{idx:<2} {gen['fullpath']:<28} "
        f"type={gen['type']:<22} tproc={gen.get('tproc_ch')} "
        f"tmux={gen.get('tmux_ch')} dac={gen.get('dac')}"
    )

## Build an AveragerProgram AWG tuning program

This uses the same user-facing style as hardware QICK programs: subclass `AveragerProgram`, configure a normal `axis_signal_gen_v6` pulse in `initialize()`, call `pulse()` for that generator in `body()`, and call `awg_ramp()` for the AWG tuning channel in the same `body()`.

In [ ]:
siggen_idx = next(idx for idx, gen in enumerate(sim["gens"]) if gen["type"] == "axis_signal_gen_v6")
awg_idx = next(idx for idx, gen in enumerate(sim["gens"]) if gen.get("gen_type") == "awg_tuning")
print("Signal generator index:", siggen_idx)
print("Signal generator config:", sim["gens"][siggen_idx])
print("AWG tuning generator index:", awg_idx)
print("AWG tuning generator config:", sim["gens"][awg_idx])


class AWGTuning_Loopback_Test(AveragerProgram):
    def initialize(self):
        self.siggen_ch = self.cfg["siggen_ch"]
        self.awg_ch = self.cfg["awg_ch"]

        # Normal axis_signal_gen_v6 output. This is a DDS-only const pulse;
        # it uses the usual QICK generator flow and is independent from AWG tuning.
        self.declare_gen(
            ch=self.siggen_ch,
            nqz=1,
        )
        self.set_pulse_registers(
            ch=self.siggen_ch,
            style="const",
            freq=self.cfg["siggen_freq_word"],
            phase=0,
            gain=self.cfg["siggen_gain"],
            length=self.cfg["siggen_length"],
            phrst=1,
            stdysel="zero",
        )

        # Initial DC-like SET value. The SET duration is used for scheduling only;
        # the AWG tuning RTL holds this value until the next command.
        self.awg_set(
            ch=self.awg_ch,
            value=self.cfg["start_value"],
            duration=self.cfg["set_duration"],
            t=20,
        )

        self.synci(100)

    def body(self):
        # Fire the normal signal generator output in the same body.
        self.pulse(
            ch=self.siggen_ch,
            t=90,
        )

        # RAMP starts from the AWG tuning internal current value and moves to target_value.
        self.awg_ramp(
            ch=self.awg_ch,
            target=self.cfg["target_value"],
            duration=self.cfg["ramp_duration"],
            t=100,
        )

        self.sync_all()


prog_cfg = {
    "reps": 1,
    "siggen_ch": siggen_idx,
    "awg_ch": awg_idx,
    "siggen_freq_word": 0x04000000,
    "siggen_gain": 30000,
    "siggen_length": 96,
    "start_value": 1000,
    "target_value": 4000,
    "set_duration": 64,
    "ramp_duration": 160,
}

prog = AWGTuning_Loopback_Test(sim, prog_cfg)
print(prog)


## Run simulation

In [ ]:
result = sim.simulate_program(prog, cycles=CYCLES)
active_output_names = [
    sim["gens"][siggen_idx]["fullpath"],
    sim["gens"][awg_idx]["fullpath"],
]
result.summary()


In [ ]:
print("RFDC outputs:")
for name, out in result.outputs.items():
    nonzero = int((out.lane_samples != 0).sum())
    valid = int(out.tvalid.sum())
    print(f"  {name:<28} dac={out.dac} lanes={out.n_lanes} valid_words={valid} nonzero_samples={nonzero}")

## Inspect samples

In [ ]:
signal_name = sim["gens"][siggen_idx]["fullpath"]
signal_out = result.outputs[signal_name]
awg_name = sim["gens"][awg_idx]["fullpath"]
awg_out = result.outputs[awg_name]

print("Signal generator output:", signal_name, "DAC", signal_out.dac)
flat_signal = result.get_output_samples(signal_name)
signal_nonzero_idx = next((idx for idx, value in enumerate(flat_signal) if value != 0), 0)
signal_window_start = max(0, signal_nonzero_idx - 16)
print("flattened signal samples around first nonzero output:")
print(flat_signal[signal_window_start:signal_window_start + 48].tolist())

print("AWG output:", awg_name, "DAC", awg_out.dac)
print("lane-0 fabric-cycle samples 0..47:")
print(awg_out.lane_samples[:48, 0].tolist())

flat_awg = result.get_output_samples(awg_name)
nonzero_idx = next((idx for idx, value in enumerate(flat_awg) if value != 0), 0)
window_start = max(0, nonzero_idx - 16)
print("flattened samples around first nonzero AWG output:")
print(flat_awg[window_start:window_start + 48].tolist())

## Plot RFDC-facing outputs

This plot uses lane-flattened digital RFDC input samples for the active `axis_signal_gen_v6` and `axis_awg_tuning_v1` outputs only. The x-axis is computed from the configured DAC sample rate.

If `matplotlib` is not installed in the selected notebook kernel, run `%pip install matplotlib` in a cell and then rerun the plot cell.

In [ ]:
import sys

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
    print("matplotlib OK")
except ModuleNotFoundError:
    HAS_MATPLOTLIB = False
    print("matplotlib is not installed in this notebook kernel.")
    print("Run this in a notebook cell, then rerun the plot cell:")
    print("%pip install matplotlib")
    print("Kernel Python:", sys.executable)

In [ ]:
if not HAS_MATPLOTLIB:
    raise RuntimeError("matplotlib is missing. Run `%pip install matplotlib` in this kernel first.")

fig, axes = result.plot_outputs(
    samples=PLOT_SAMPLES,
    channels=active_output_names,
    show=False,
)
fig


## Optional CSV export

In [ ]:
SAVE_CSV = False
CSV_PREFIX = REPO / "qick" / "qick_demos" / "qicksim_output_waveform_demo"

if SAVE_CSV:
    csv_paths = result.to_csv(CSV_PREFIX, overwrite=True)
    print(csv_paths)
else:
    print("CSV export disabled. Set SAVE_CSV=True to write files.")